## Basics of Numerical Integration

This notebook introduces the basics of numerical integration and some implementation in Julia.

Two big approaches:
1. Newton-Cotes - local polynomial approximations
2. Quadrature - global polynomial approximations

Quadrature is vastly superior for well-behaved functions so we will mainly focus on that

### Quadrature
Idea: for any $n$ there exists $n$ nodes and weights such that

$$ \int_{-1}^1 f(x) dx = \sum_{i=1}^n w_i f(x_i) $$

holds exactly if $f$ is a polynomial of degree $2n - 1$. 

In Julia, we have three alternatives:
1. Adaptive quadrature
    - You don't have to specify the nodes or weights. Implemented in `QuadGK`
    - Slow, not well suited to inner loops
2. Gaussian Quadrature
    - Implented in `FastGaussQuadrature.jl`
    - Non adaptive becase you tell the computer the number of nodes
3. Expectations
    - If calculation is simply for calculating expectations of a particular distribution, `Expectations.jl` can be useful
    
### Adaptive quadrature
We just pass the function we want to integrate over and the integral bounds. Here is an example calculating

$$ \int_{-2\pi}^{2\pi} cos(x)dx $$

In [2]:
using QuadGK
value, tol = quadgk(cos, -2π, 2π)

(-1.5474478810961125e-14, 5.7846097329025695e-24)

### FastGauss Quadrature
Since it is not adaptive, way more efficient. Only caveat is that we have to remap from the region we are integrating to the interval $[-1,1]$. The function `gausslegendre` gives you the weights and nodes already on that domain. 

In [162]:
using FastGaussQuadrature, LinearAlgebra
x, w = gausslegendre(100_000);

Now let's find the integral of $f(x)$ from $-2$ to $2$. To do this we will use the mapping that takes any $x\in[-1,1]$ and projects to $[a,b]$:

$$ \left( \frac{b-a}{2} \right)(x +1) + a $$

In [163]:
function map(x, a, b)
    ((b-a)/2)*(x + 1) + a
end

map (generic function with 2 methods)

In [ ]:
function map2(x, a, b)
    ((b-a)/2)*x + (b+a)/2
end

In [164]:
a = -3
b = 2
x = map.(x, a, b);

Now given our $x$, we compute the integral as:
$$ f(x) = \frac{b-a}{2} \sum_i w_i f(x_i) $$
because we are using the change of variables $y = \frac{2}{b-a}(x-a) - 1$ to map the nodes from $[-1,1]$ to $[a,b]$:
$$ \int_a^b f(x) dx = \frac{b-a}{2} \int_{-1}^1 f \left( \frac{b-a}{2}(y+1) + a\right) dy $$
So we evaluate $f$ at the mapped nodes, take the weighted sum, and multiply by $\frac{b-a}{2}$.

In [165]:
f(x) = x^2
(b-a)/2 * (w ⋅ f.(x))

11.666666666666664

Let's now write a function that computes the integral given function $f$ and interval of interest

In [161]:
function quad_int(f, a, b, N)
    x, w = gausslegendre(N);
    x = map.(x, a, b);
    return (b-a)/2 * (w ⋅ f.(x))
end

quad_int (generic function with 1 method)

`QuantEcon` has a routine that automates the mapping step so we can use it to check

In [168]:
using QuantEcon
x, w = qnwlege(10, -3, 2)
w ⋅ f.(x)

11.666666666666531

In [169]:
quad_int((x) -> x^2, -3, 2, 1000)

11.666666666666664